<a href="https://colab.research.google.com/github/luciapola04/progetto_RO/blob/main/progetto_RO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Lettura del Problema (Parsing)

Prima di poter applicare qualsiasi teorema matematico, dobbiamo tradurre il problema di Programmazione Lineare (PL) da un formato leggibile per l'uomo a uno comprensibile per la macchina.

Un problema di PL standard è definito da:
* Una **Funzione Obiettivo (F.O.)** da massimizzare o minimizzare: $z = c^T x$
* Un set di **Vincoli**: $Ax \le b$, $Ax \ge b$, oppure $Ax = b$

La funzione `parse_equazione` utilizza le **Espressioni Regolari (Regex)** per estrarre i coefficienti ($c$ per la F.O., la matrice $A$ per i vincoli), i termini noti (il vettore $b$) e la direzione delle disuguaglianze. Questo ci permette di inserire le equazioni come semplici stringhe di testo.

In [12]:
import re

def parse_equazione(equazione):
    eq = equazione.replace(" ", "")

    if "<=" in eq:
        operatore = "<="
        sinistra, destra = eq.split("<=")
    elif ">=" in eq:
        operatore = ">="
        sinistra, destra = eq.split(">=")
    elif "=" in eq:
        operatore = "="
        sinistra, destra = eq.split("=")
    else:
        raise ValueError(f"Nessun operatore valido trovato in: {equazione}")

    termine_noto = float(destra)
    pattern = r'([+-]?\d*)([a-zA-Z]+[0-9]*)?'
    matches = re.finditer(pattern, sinistra)
    coefficienti = {}

    for m in matches:
        coeff, var = m.groups()
        if coeff == "" and var is None:
            continue
        if var is None:
            termine_noto -= float(coeff)
            continue
        if coeff in ["", "+"]:
            coeff = 1.0
        elif coeff == "-":
            coeff = -1.0
        else:
            coeff = float(coeff)
        coefficienti[var] = coefficienti.get(var, 0) + coeff

    return coefficienti, operatore, termine_noto

Inserire nella cella seguente il PL e la soluzione da verificare:

In [13]:
# --- DEFINIZIONE DEL PROBLEMA ---
tipo_problema = "max"

# Funzione obiettivo
funzione_obiettivo = "2x1 + 1x2 = 0"

# Tre vincoli che si intersecano tutti nel punto x1=2, x2=2 (Caso Degenere)
vincoli_test = [
    "1x1 + 0x2 <= 2",
    "0x1 + 1x2 <= 2",
    "2x1 + 2x2 <= 8"
]

soluzione_ottima = {"x1": 2.0, "x2": 2.0}

Eseguire la cella seguente per estrarre i dati tramite la funzione `parse_equazione`

In [14]:
variabili_totali = set()

# Estrazione Dati
coeffs_fo, op_fo, c_fo = parse_equazione(funzione_obiettivo)
variabili_totali.update(coeffs_fo.keys())

dati_vincoli = []
print("--- Vincoli Analizzati ---")
for i, equazione_vincolo in enumerate(vincoli_test):
    coeffs, operatore, termine_noto = parse_equazione(equazione_vincolo)
    dati_vincoli.append({
        "coefficienti": coeffs,
        "operatore": operatore,
        "termine_noto": termine_noto
    })
    variabili_totali.update(coeffs.keys())
    print(f"Vincolo {i+1}: {coeffs} {operatore} {termine_noto}")

print(f"\nVariabili nel sistema: {sorted(variabili_totali)}")
print(f"Soluzione in esame: {soluzione_ottima}")

--- Vincoli Analizzati ---
Vincolo 1: {'x1': 1.0, 'x2': 0.0} <= 2.0
Vincolo 2: {'x1': 0.0, 'x2': 1.0} <= 2.0
Vincolo 3: {'x1': 2.0, 'x2': 2.0} <= 8.0

Variabili nel sistema: ['x1', 'x2']
Soluzione in esame: {'x1': 2.0, 'x2': 2.0}


## 2. Verifica di Ammissibilità e Calcolo degli Scarti

Il Teorema degli Scarti Complementari ha senso solo se il punto candidato si trova all'interno della regione ammissibile (ovvero rispetta tutti i vincoli).

Per ogni vincolo $i$, calcoliamo la differenza tra il lato sinistro (LHS, i valori delle variabili moltiplicati per i coefficienti) e il lato destro (RHS, il termine noto). Questa differenza è lo **scarto** ($s_i$).

* Se $s_i = 0$, il vincolo si dice **attivo** (la soluzione giace esattamente sul bordo di quel vincolo).
* Se $s_i > 0$, il vincolo è **non attivo** (la soluzione è all'interno della regione rispetto a quel bordo).
* Se il vincolo viene violato, la soluzione non è ammissibile e l'algoritmo deve fermarsi.

In [8]:
scarti_primali = []
ammissibile = True

print("--- Verifica Ammissibilità Primale e Calcolo Scarti ---\n")

for i, vincolo in enumerate(dati_vincoli):
    coeffs = vincolo["coefficienti"]
    operatore = vincolo["operatore"]
    rhs = vincolo["termine_noto"]

    lhs = 0.0
    for var, valore in soluzione_ottima.items():
        lhs += coeffs.get(var, 0.0) * valore

    scarto = 0.0
    vincolo_rispettato = False

    if operatore == "<=":
        if lhs <= rhs + 1e-6: # Tolleranza per float
            vincolo_rispettato = True
            scarto = max(0.0, rhs - lhs)

    elif operatore == ">=":
        if lhs >= rhs - 1e-6:
            vincolo_rispettato = True
            scarto = max(0.0, lhs - rhs)

    elif operatore == "=":
        if abs(lhs - rhs) < 1e-6:
            vincolo_rispettato = True
            scarto = 0.0

    print(f"Vincolo {i+1}: LHS = {lhs} | Operatore: {operatore} | RHS = {rhs}")

    if vincolo_rispettato:
        print(f"  ✓ Rispettato. Scarto (s{i+1}) = {scarto}")
        scarti_primali.append(scarto)
    else:
        print(f"  X NON RISPETTATO! (Violato di {abs(rhs - lhs)})")
        ammissibile = False
        scarti_primali.append(None)

print("-" * 45)
if ammissibile:
    print("ESITO: La soluzione è PRIMALE AMMISSIBILE! Procediamo con il duale.")
else:
    print("ESITO: La soluzione NON E' AMMISSIBILE! L'algoritmo si ferma qui.")

--- Verifica Ammissibilità Primale e Calcolo Scarti ---

Vincolo 1: LHS = 2.0 | Operatore: <= | RHS = 2.0
  ✓ Rispettato. Scarto (s1) = 0.0
Vincolo 2: LHS = 2.0 | Operatore: <= | RHS = 2.0
  ✓ Rispettato. Scarto (s2) = 0.0
Vincolo 3: LHS = 8.0 | Operatore: <= | RHS = 8.0
  ✓ Rispettato. Scarto (s3) = 0.0
---------------------------------------------
ESITO: La soluzione è PRIMALE AMMISSIBILE! Procediamo con il duale.


## 3. Il Teorema degli Scarti Complementari

Se la soluzione primale $x^*$ è ottima, deve esistere un vettore duale $w^*$ ammissibile che rispetta due regole fondamentali (gli scarti complementari):

1. **Regola dei vincoli primali:** Se lo scarto primale $s_i > 0$ (vincolo non attivo), allora la corrispondente variabile duale $w_i = 0$. Se $s_i = 0$, $w_i$ può essere non nulla.
2. **Regola delle variabili primali:** Se una variabile primale $x_j > 0$, allora il corrispondente vincolo duale deve essere *attivo* (ovvero, deve valere l'uguaglianza stretta: $\sum A_{ij} w_i = c_j$).

Costruiamo questo sistema di equazioni e disequazioni. Poiché i vertici degeneri possono creare sistemi sovradeterminati ma coerenti, utilizziamo `scipy.optimize.linprog`. Non ci interessa minimizzare una funzione (usiamo un vettore $c$ di zeri), ci basta che `linprog` trovi *almeno un* punto ammissibile nel poliedro duale generato da queste regole. Se lo trova, la soluzione primale è ottima!

In [15]:
import numpy as np
from scipy.optimize import linprog

# Eseguiamo questa parte SOLO se la soluzione primale è risultata ammissibile
if ammissibile:
    print("--- Teorema degli Scarti Complementari ---\n")

    num_vincoli = len(dati_vincoli)
    w_nomi = [f"w{i+1}" for i in range(num_vincoli)]
    bounds_w = []

    print("1. Deduzioni dagli Scarti Primali (Regola 1) e Segni Duali:")
    for i, vincolo in enumerate(dati_vincoli):
        scarto = scarti_primali[i]
        op = vincolo["operatore"]

        if scarto > 1e-6:
            bounds_w.append((0.0, 0.0))
            print(f"  s{i+1} > 0  --> {w_nomi[i]} = 0")
        else:
            if tipo_problema == "max":
                if op == "<=": bounds_w.append((0, None))
                elif op == ">=": bounds_w.append((None, 0))
                else: bounds_w.append((None, None))
            elif tipo_problema == "min":
                if op == ">=": bounds_w.append((0, None))
                elif op == "<=": bounds_w.append((None, 0))
                else: bounds_w.append((None, None))
            print(f"  s{i+1} = 0  --> {w_nomi[i]} è incognita con bound {bounds_w[-1]}")

    print("\n2. Costruzione dei Vincoli Duali (Regola 2):")
    A_eq, b_eq = [], []
    A_ub, b_ub = [], []

    variabili_ordinate = sorted(list(variabili_totali))

    for var in variabili_ordinate:
        valore_x = soluzione_ottima.get(var, 0)
        c_j = coeffs_fo.get(var, 0.0)

        colonna_A = [vincolo["coefficienti"].get(var, 0.0) for vincolo in dati_vincoli]

        if valore_x > 1e-6:
            A_eq.append(colonna_A)
            b_eq.append(c_j)
            print(f"  {var} > 0 --> Vincolo duale di {var} forzato a UGUAGLIANZA (= {c_j})")
        else:
            if tipo_problema == "max":
                A_ub.append([-a for a in colonna_A])
                b_ub.append(-c_j)
            elif tipo_problema == "min":
                A_ub.append(colonna_A)
                b_ub.append(c_j)
            print(f"  {var} = 0 --> Vincolo duale di {var} resta DISEGUAGLIANZA")

    A_eq = np.array(A_eq) if A_eq else None
    b_eq = np.array(b_eq) if b_eq else None
    A_ub = np.array(A_ub) if A_ub else None
    b_ub = np.array(b_ub) if b_ub else None

    print("\n3. Risoluzione tramite Ricerca Operativa:")
    c_dummy = np.zeros(num_vincoli)

    risultato = linprog(c_dummy, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds_w, method='highs')

    print("\n" + "=" * 55)
    if risultato.success:
        print("★★★ VERDETTO: LA SOLUZIONE INSERITA E' OTTIMA! ★★★")
        print("Il sistema duale ammette almeno una soluzione valida:")
        for i, w_val in enumerate(risultato.x):
            print(f"  {w_nomi[i]} = {round(w_val, 4)}")
    else:
        print("⚠ VERDETTO: LA SOLUZIONE NON E' OTTIMA!")
        print("Impossibile trovare variabili duali ammissibili (sistema duale incompatibile).")
    print("=" * 55)
else:
    print("Esecuzione saltata: la soluzione primale non è ammissibile.")

--- Teorema degli Scarti Complementari ---

1. Deduzioni dagli Scarti Primali (Regola 1) e Segni Duali:
  s1 = 0  --> w1 è incognita con bound (0, None)
  s2 = 0  --> w2 è incognita con bound (0, None)
  s3 = 0  --> w3 è incognita con bound (0, None)

2. Costruzione dei Vincoli Duali (Regola 2):
  x1 > 0 --> Vincolo duale di x1 forzato a UGUAGLIANZA (= 2.0)
  x2 > 0 --> Vincolo duale di x2 forzato a UGUAGLIANZA (= 1.0)

3. Risoluzione tramite Ricerca Operativa:

★★★ VERDETTO: LA SOLUZIONE INSERITA E' OTTIMA! ★★★
Il sistema duale ammette almeno una soluzione valida:
  w1 = 1.0
  w2 = 0.0
  w3 = 0.5
